# Training a Superhuman, 100% Explainable Bridge Bidding System from Scratch (Colab GPU)

This notebook walks step-by-step through training a **superhuman, 100% explainable Bridge bidding system from scratch** on a Google Colab GPU (`T4` / `L4` / `A100`), combining **all 10 methods** in the [`ed2k/bid`](https://github.com/ed2k/bid) repository:

1. **152-Feature Bitmask Hand, `SuitC`, & Sequence-Exact Convention State Engine** (`src/bid/features.py`, `src/bid/suitc.py`)
2. **Declarative Symbolic `.dsl` Rules & Priority/Veto `DecisionNet`** (`src/bid/decision_net.py`, `src/bid/diagnostics.py`)
3. **Lockstep GPU-Vectorized Arena (`TensorDecisionNet`, `BatchedAuctionEnv`, `DealBundle`) — up to 16,800 boards/sec** (`research/gpu_arena.py`)
4. **GPU Counterfactual Continuation Q-Tables (`gpu_continuation_q_values`) & ID3 Rule Distillation** (`src/bid/learner.py`, `research/brill_distill.py`)
5. **Sequence-Exact Convention & Slam Grafting (`conv_state`, Jacoby `2NT`, `RKCB 0314`)** (`research/graft_conventions.py`)
6. **Interactive DAgger Off-Policy Harvesting & DD Counterfactual Filtering** (`research/brill_dagger_dd.py`)
7. **Neuro-Symbolic Differentiable DSL (`DiffDSLNet`) with Straight-Through Estimator (`STE`) Gradients & 100% Explainability** (`src/bid/diff_dsl_net.py`)
8. **Multi-Generation GPU Joint Policy Search (`JPS`): Depth-2 Coupled Pairs, `alterReach`, Empirical Bayes Shrinkage, STE Thresholds & Partner-Belief Splits** (`research/jps_search.py`)
9. **Test-Time Bayesian Single-Dummy Rollout Search (`BayesianSingleDummyBidder`)** (`src/bid/bayesian_search.py`)
10. **Optional Extension: Chain-of-Thought Transformer Student (`COTModel`) & `REINFORCE` Self-Play** (`src/bid/cot_model.py`, `src/bid/rl_finetune.py`)

> **Colab Runtime Tip**: Select **Runtime -> Change runtime type -> `L4` GPU** (`1.94 CU/hr`, best cost/performance for `B = 10,000+` JPS) or **`T4` GPU** (`1.15 CU/hr`).


## Stage 0 — Colab GPU Environment Setup & Repository Check

This cell works identically **on Google Colab** (cloning `ed2k/bid` if needed and verifying PyTorch CUDA + optional Linux `libdds.so` compilation) and **inside a local clone**. All temporary scratch artifacts are written to `/tmp/colab_bid_scratch/` so tracked repository weights are never overwritten.


In [ ]:
import os
import sys
import subprocess

# 1. Locate or clone the repository on Colab
if not os.path.exists("src/bid") and os.path.exists("/content"):
    repo_dir = "/content/bid"
    if not os.path.exists(repo_dir):
        subprocess.run(["git", "clone", "https://github.com/ed2k/bid.git", repo_dir], check=True)
    os.chdir(repo_dir)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

for p in ("src", "research"):
    abs_p = os.path.abspath(p)
    if abs_p not in sys.path:
        sys.path.insert(0, abs_p)

# 2. On Linux Colab, optionally build libdds.so so DDSolver runs natively at full C++ speed
if sys.platform.startswith("linux") and not os.path.exists("bin/libdds.so"):
    print("Linux detected without bin/libdds.so — precomputed DealBundles work immediately without it.")
    print("To build fresh Double-Dummy deals from scratch on Colab, uncomment and run:")
    print("  !sudo apt-get update -qq && sudo apt-get install -y -qq libomp-dev")
    print("  !git clone --depth 1 https://github.com/dds-bridge/dds.git /tmp/dds && cd /tmp/dds/src && g++ -O3 -fPIC -shared -fopenmp -std=c++17 *.cpp -o " + os.path.abspath("bin/libdds.so"))

import torch
import gpu_arena as ga

DEVICE = ga.pick_device()
WORK_DIR = "/tmp/colab_bid_scratch"
os.makedirs(WORK_DIR, exist_ok=True)

print(f"Repo root    : {os.getcwd()}")
print(f"PyTorch      : {torch.__version__}")
print(f"Compute dev  : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU Model    : {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")
print(f"Feature keys : {ga.NUM_FEATURES} canonical BridgeFeatures")
print(f"Work dir     : {WORK_DIR}")


## Stage 1 — Precomputing vs. Loading a GPU `DealBundle` (Static Features + 20-Entry Double-Dummy Tables)

Why does [`research/gpu_arena.py`](research/gpu_arena.py) hit **16,800 boards/sec** on an A100?
1. **Static vs. Dynamic Feature Split**: ~90 of the `152` features depend only on a player's 13 cards (`hcp`, `losing_trick_count`, `quick_tricks`, `SuitC` intermediates, `*_realsolid`). They are precomputed **once per deal** into `hand_features: (B, 4, 152)`.
2. **20-Entry Double-Dummy Lookup**: Each board's `5 strains x 4 declarers` optimal trick counts are stored in `dd_tricks: (B, 4, 5)` alongside exact suit honors/stoppers, eliminating `libdds` C++ FFI overhead during millions of GPU rollouts.

Below we generate a small fresh bundle from random deals **and** load the repository's pre-solved benchmark bundle (`data/colab_bundle_10k.pt` or `data/colab_bundle_500.pt`) onto the GPU.


In [ ]:
import time
from bid.eval_vs_dds import build_deals

# 1A. Build a fresh 32-board DealBundle from scratch (uses DDSolver.solve_dd_table)
t0 = time.perf_counter()
fresh_deals = build_deals(32, seed=123, include_stratified=False)
fresh_bundle = ga.DealBundle.from_deals(fresh_deals, compute_dd=True, device=DEVICE)
print(f"Built fresh 32-board DealBundle on {DEVICE} in {time.perf_counter() - t0:.2f}s | dd_tricks={tuple(fresh_bundle.dd_tricks.shape)}")

# 1B. Load precomputed benchmark bundle for large-scale GPU training
if os.path.exists("data/colab_bundle_10k.pt") and DEVICE.type == "cuda":
    bundle_path = "data/colab_bundle_10k.pt"
elif os.path.exists("data/colab_bundle_10k.pt"):
    bundle_path = "data/colab_bundle_10k.pt"
else:
    bundle_path = "data/colab_bundle_500.pt"

full_bundle = ga.DealBundle.load(bundle_path, device=DEVICE)
print(f"Loaded {bundle_path}: {full_bundle.num_boards} boards on {DEVICE}")
print(f"Tensors: hand_features={tuple(full_bundle.hand_features.shape)}, dd_tricks={tuple(full_bundle.dd_tricks.shape)}")


## Stage 2 — Writing a Minimal Seed Bidding System from Scratch (`scratch_seed.dsl`)

Every superhuman run starts from a small, readable **warm-start policy**. Here we construct a clean **22-rule Natural 5-Card Major seed system** purely from Python `DecisionNetRule` primitives (1-level openings, `1NT`, weak twos, simple raises, game raises, 1-level overcalls, and takeout doubles), export it as `scratch_seed.dsl`, and verify **0-mismatch bit-exact parity** between CPU `DecisionNet` and GPU `TensorDecisionNet`.


In [ ]:
from bid.decision_net import DecisionNet, DecisionNetRule, RuleCondition
from bid.diff_dsl_net import DiffDSLNet
from bid.eval_vs_dds import parse_call

def build_scratch_seed_net() -> DecisionNet:
    net = DecisionNet("scratch_seed")
    specs = [
        # Openings (1-level & 1NT & Weak Twos)
        ("SEED_OPEN_1NT", "1NT", 30, [("is_opening", "==", True), ("is_balanced", "==", True), ("hcp", ">=", 15), ("hcp", "<=", 17)]),
        ("SEED_OPEN_1S",  "1S",  25, [("is_opening", "==", True), ("hcp", ">=", 12), ("spade_len", ">=", 5), ("s_is_best_major", "==", True)]),
        ("SEED_OPEN_1H",  "1H",  24, [("is_opening", "==", True), ("hcp", ">=", 12), ("heart_len", ">=", 5), ("spade_len", "<=", 4)]),
        ("SEED_OPEN_1D",  "1D",  20, [("is_opening", "==", True), ("hcp", ">=", 12), ("spade_len", "<=", 4), ("heart_len", "<=", 4), ("diamond_len", ">=", 4), ("d_is_best_minor", "==", True)]),
        ("SEED_OPEN_1C",  "1C",  19, [("is_opening", "==", True), ("hcp", ">=", 12), ("spade_len", "<=", 4), ("heart_len", "<=", 4)]),
        ("SEED_OPEN_2S",  "2S",  18, [("is_opening", "==", True), ("hcp", ">=", 6), ("hcp", "<=", 10), ("spade_len", "==", 6), ("s_top3_honors", ">=", 1)]),
        ("SEED_OPEN_2H",  "2H",  17, [("is_opening", "==", True), ("hcp", ">=", 6), ("hcp", "<=", 10), ("heart_len", "==", 6), ("h_top3_honors", ">=", 1)]),
        ("SEED_OPEN_PASS","PASS", 1, [("is_opening", "==", True)]),

        # Simple Responses to Partner's 1-level Openings
        ("SEED_RESP_1NT_3NT", "3NT", 35, [("opponents_bid", "==", False), ("partner_last_call", "==", "1NT"), ("my_last_call", "==", "NONE"), ("hcp", ">=", 10)]),
        ("SEED_RESP_1NT_2NT", "2NT", 30, [("opponents_bid", "==", False), ("partner_last_call", "==", "1NT"), ("my_last_call", "==", "NONE"), ("hcp", ">=", 8), ("hcp", "<=", 9)]),
        ("SEED_RESP_4S", "4S", 34, [("opponents_bid", "==", False), ("partner_last_call", "==", "1S"), ("my_last_call", "==", "NONE"), ("spade_len", ">=", 4), ("hcp", ">=", 12)]),
        ("SEED_RESP_2S", "2S", 28, [("opponents_bid", "==", False), ("partner_last_call", "==", "1S"), ("my_last_call", "==", "NONE"), ("spade_len", ">=", 3), ("hcp", ">=", 6), ("hcp", "<=", 10)]),
        ("SEED_RESP_4H", "4H", 34, [("opponents_bid", "==", False), ("partner_last_call", "==", "1H"), ("my_last_call", "==", "NONE"), ("heart_len", ">=", 4), ("hcp", ">=", 12)]),
        ("SEED_RESP_2H", "2H", 28, [("opponents_bid", "==", False), ("partner_last_call", "==", "1H"), ("my_last_call", "==", "NONE"), ("heart_len", ">=", 3), ("hcp", ">=", 6), ("hcp", "<=", 10)]),
        ("SEED_RESP_1S_OVER_1H", "1S", 26, [("opponents_bid", "==", False), ("partner_last_call", "in", ["1C", "1D", "1H"]), ("my_last_call", "==", "NONE"), ("spade_len", ">=", 4), ("hcp", ">=", 6)]),
        ("SEED_RESP_1H_OVER_m",  "1H", 25, [("opponents_bid", "==", False), ("partner_last_call", "in", ["1C", "1D"]), ("my_last_call", "==", "NONE"), ("heart_len", ">=", 4), ("hcp", ">=", 6)]),
        ("SEED_RESP_1NT", "1NT", 20, [("opponents_bid", "==", False), ("partner_last_call", "in", ["1C", "1D", "1H", "1S"]), ("my_last_call", "==", "NONE"), ("hcp", ">=", 6), ("hcp", "<=", 10)]),

        # Opener Continuations
        ("SEED_OPEN_REBID_3NT", "3NT", 30, [("my_last_call", "in", ["1C", "1D", "1H", "1S"]), ("combined_hcp_min", ">=", 25), ("is_balanced", "==", True)]),
        ("SEED_OPEN_REBID_4S",  "4S",  32, [("my_last_call", "==", "1S"), ("partner_last_call", "==", "2S"), ("hcp", ">=", 17)]),
        ("SEED_OPEN_REBID_4H",  "4H",  32, [("my_last_call", "==", "1H"), ("partner_last_call", "==", "2H"), ("hcp", ">=", 17)]),

        # Basic Competitive Overcalls & Doubles
        ("SEED_OVERCALL_1NT", "1NT", 30, [("is_opening", "==", False), ("my_last_call", "==", "NONE"), ("last_bid_level", "==", 1), ("is_balanced", "==", True), ("hcp", ">=", 15), ("hcp", "<=", 18), ("has_stopper", "==", True)]),
        ("SEED_OVERCALL_1S",  "1S",  26, [("is_opening", "==", False), ("my_last_call", "==", "NONE"), ("last_bid_level", "==", 1), ("opp_bid_spades", "==", False), ("spade_len", ">=", 5), ("hcp", ">=", 8)]),
        ("SEED_OVERCALL_1H",  "1H",  25, [("is_opening", "==", False), ("my_last_call", "==", "NONE"), ("last_bid_level", "==", 1), ("opp_bid_hearts", "==", False), ("heart_len", ">=", 5), ("hcp", ">=", 8)]),
        ("SEED_COMP_X",       "X",   24, [("is_opening", "==", False), ("my_last_call", "==", "NONE"), ("last_bid_level", "==", 1), ("hcp", ">=", 13)]),
    ]
    for rid, call_s, prio, conds in specs:
        net.add_rule(DecisionNetRule(rid, parse_call(call_s), [RuleCondition(k, op, v) for k, op, v in conds], priority=prio))
    return net

seed_net = build_scratch_seed_net()
seed_dsl_path = os.path.join(WORK_DIR, "scratch_seed.dsl")
DiffDSLNet(seed_net, device=DEVICE).export_dsl(seed_dsl_path)

# Verify 0-mismatch parity vs Python DecisionNet
checked, miss = ga.verify_parity(fresh_deals[:16], seed_dsl_path, device=DEVICE)
print(f"Exported {len(seed_net.rules)}-rule seed system -> {seed_dsl_path}")
print(f"CPU <-> GPU parity check (16 boards): {checked} decisions, {miss} mismatches")

# Fast GPU match against legacy baseline champion_system.dsl
seed_t = ga.TensorDecisionNet.from_dsl(seed_dsl_path, device=DEVICE)
eval_b = full_bundle if full_bundle.num_boards <= 2500 else ga.DealBundle(
    full_bundle.hand_features[:1000], full_bundle.dealers[:1000], full_bundle.vulns[:1000],
    full_bundle.dd_tricks[:1000], full_bundle.suit_lengths[:1000], full_bundle.suit_has_king[:1000],
    full_bundle.suit_has_queen[:1000], full_bundle.suit_stopper_q[:1000], full_bundle.ace_counts[:1000],
    full_bundle.suit_intermediates[:1000],
)
if os.path.exists("system/champion_system.dsl"):
    champ_t = ga.TensorDecisionNet.from_dsl("system/champion_system.dsl", device=DEVICE)
    res_seed = ga.gpu_team_match(eval_b, seed_t, champ_t)
    print(f"Stage 2 Seed vs champion_system ({eval_b.num_boards} bds, {res_seed['elapsed_ms']:.1f}ms): {res_seed['mean']:+.3f} IMP/bd (t={res_seed['t']:+.2f})")


## Stage 3 — Bootstrapping New Rules via GPU Counterfactual Q-Tables & ID3 Distillation

How do you discover **hundreds of new rules from scratch** without relying on an external teacher?
Use **Monte Carlo / Counterfactual Double-Dummy Continuation Rollouts** (`gpu_continuation_q_values`):
1. For every decision state in a batch of $B$ boards, force each legal candidate call $c \in \{0, \dots, 37\}$ simultaneously on the GPU.
2. Roll out the rest of the auction under the current policy `net`.
3. Label each position with its **highest-EV legal call** `argmax_c Q(s, c)` and fit an **`ID3DecisionTree`** (`src/bid/learner.py`), compiling every leaf path back into ordinary `.dsl` rules via `id3_tree_to_rules(...)`.

*(Note: When training against an existing oracle like Remote Brill, [`research/brill_distill.py`](research/brill_distill.py) uses this exact `ID3DecisionTree -> id3_tree_to_rules -> merge_dsl.py` pipeline with `--relabel outcome` across `330,000` traces to produce `system/brill_distilled.dsl`.)*


In [ ]:
import jps_search as jps
from bid.learner import ID3DecisionTree, id3_tree_to_rules
from bid.models import CallType

# Slice training sub-bundle for Q-table rollouts
n_q = min(256 if DEVICE.type == "cuda" else 128, full_bundle.num_boards)
q_bundle = jps.slice_bundle(full_bundle, torch.arange(n_q, device=DEVICE))

# Build Turn-0 counterfactual Q-table across all legal opening actions (38 calls x n_q boards)
t0 = time.perf_counter()
q_points, q_lmask = ga.gpu_continuation_q_values(q_bundle, seed_t)
dt_q = time.perf_counter() - t0
q_masked = torch.where(q_lmask & (~torch.isnan(q_points)), q_points, torch.full_like(q_points, -1e9))
best_q_vals, best_c_idx = q_masked.max(dim=1)
print(f"Computed ({n_q} x 38) GPU Continuation Q-Table in {dt_q:.2f}s")

# Convert tensor features to dicts and fit an ID3DecisionTree to discover refined opening/preempt rules
env_q = ga.BatchedAuctionEnv(q_bundle)
X_q, _ = env_q.build_features()
id3_keys = [
    "hcp", "total_points", "spade_len", "heart_len", "diamond_len", "club_len",
    "longest_suit_len", "second_longest_len", "is_balanced", "is_semi_balanced",
    "controls", "quick_tricks", "losing_trick_count", "rule20_total", "rule_of_21",
    "is_favorable_vuln", "first_round_controls", "second_round_controls",
    "s_is_longest", "h_is_longest", "d_is_longest", "c_is_longest",
    "long_suit_intermediates", "short_suit_wasted_hcp",
]
id3_cols = [(k, ga.FEAT_TO_COL[k]) for k in id3_keys]

rows_q = []
labels_q = []
for b in range(n_q):
    row = {}
    for k, col in id3_cols:
        v = float(X_q[b, col].item())
        row[k] = bool(v > 0.5) if k.startswith(("is_", "rule_of_")) or k.endswith("_longest") else v
    rows_q.append(row)
    c_name = ga.CALL_NAMES[int(best_c_idx[b].item())]
    # Restrict ID3 targets to sound <=3 level calls to avoid single-deal DD variance on 5/6-level jumps
    c_obj = parse_call(c_name)
    if c_obj.type == CallType.BID and c_obj.level > 3:
        c_obj = parse_call("PASS")
    labels_q.append(c_obj)

tree = ID3DecisionTree(max_depth=4, min_samples_split=8)
tree.fit(rows_q, labels_q, candidate_features=id3_keys)
q_rules = id3_tree_to_rules(
    tree,
    intersection_rules=[],
    base_id="SCRATCH_Q_OPEN",
    base_priority=22,
    description="Discovered from GPU Continuation Q-Table",
)
# Gate learned opening rules by is_opening == True
for r in q_rules:
    r.conditions.insert(0, RuleCondition("is_opening", "==", True))

stage3_net = seed_net.clone()
for r in q_rules:
    stage3_net.add_rule(r)
stage3_dsl_path = os.path.join(WORK_DIR, "scratch_stage3.dsl")
DiffDSLNet(stage3_net, device=DEVICE).export_dsl(stage3_dsl_path)

s3_t = ga.TensorDecisionNet.from_dsl(stage3_dsl_path, device=DEVICE)
res_s3 = ga.gpu_team_match(eval_b, s3_t, seed_t)
print(f"ID3 extracted {len(q_rules)} new rules -> total {len(stage3_net.rules)} rules ({stage3_dsl_path})")
print(f"Stage 3 (Seed + Q-Table ID3) vs Stage 2 Seed: {res_s3['mean']:+.3f} IMP/bd (t={res_s3['t']:+.2f}, total={res_s3['total']:+d} IMPs)")


## Stage 4 — Sequence-Exact Convention/Slam Grafting & DAgger Competitive Alignment

From Stage 3 onward, a realistic **from-scratch OR warm-started** superhuman pipeline must fix two structural failure modes observed against world-class opponents (such as Live Remote Brill):
1. **Turn-3+ Convention Blindness** -> Fixed by [`graft_conventions.py`](research/graft_conventions.py), using our `152`-feature `conv_state` state machine (`Jacoby 2NT`, `RKCB 0314`, `5NT` King Ask, competitive advancer raises).
2. **Off-Policy Competitive Drift** -> Fixed by [`brill_dagger_dd.py`](research/brill_dagger_dd.py), harvesting disagreement trajectories (`data/traces/dagger_dd_harvest.jsonl`) and filtering fixes against Double-Dummy outcomes (`1NT` overcalls + `2NT/3NT`, takeout doubles over `1D` and `3H`, 5-level penalty doubles, and weak opponent-suit overcall guards).

Below we graft these conventions onto either our `scratch_stage3.dsl` **or** the distilled base `system/brill_gpu_evolved.dsl` to prepare a high-strength neuro-symbolic policy for `DiffDSLNet` + `JPS` self-improvement.


In [ ]:
from graft_conventions import build_program1_rules
from brill_dagger_dd import build_program2_dagger_rules
from bid.eval_vs_dds import load_decision_net_dsl

# Use full distilled/evolved backbone if available for superhuman GPU training, else scratch Stage 3
base_for_jps_path = "system/brill_gpu_evolved.dsl" if os.path.exists("system/brill_gpu_evolved.dsl") else stage3_dsl_path
stage4_net = load_decision_net_dsl(base_for_jps_path)
existing_ids = {r.rule_id for r in stage4_net.rules}

added_p1 = 0
for r in build_program1_rules():
    if r.rule_id not in existing_ids:
        stage4_net.add_rule(r)
        existing_ids.add(r.rule_id)
        added_p1 += 1

added_p2 = 0
for r in build_program2_dagger_rules():
    if r.rule_id not in existing_ids:
        stage4_net.add_rule(r)
        existing_ids.add(r.rule_id)
        added_p2 += 1

stage4_dsl_path = os.path.join(WORK_DIR, "scratch_stage4_conv.dsl")
DiffDSLNet(stage4_net, device=DEVICE).export_dsl(stage4_dsl_path)
s4_t = ga.TensorDecisionNet.from_dsl(stage4_dsl_path, device=DEVICE)

res_s4_seed = ga.gpu_team_match(eval_b, s4_t, seed_t)
print(f"Base: {base_for_jps_path} + grafted {added_p1} P1 + {added_p2} P2 rules -> {len(stage4_net.rules)} rules")
print(f"Stage 4 vs Stage 2 Seed ({eval_b.num_boards} bds): {res_s4_seed['mean']:+.3f} IMP/bd (t={res_s4_seed['t']:+.2f})")


## Stage 5 — Compiling `.dsl` into `DiffDSLNet` (`nn.Module`) + Straight-Through Estimator (`STE`) Gradients & 100% Explainability

[`DiffDSLNet`](src/bid/diff_dsl_net.py) compiles every rule and unique condition atom in `stage4_dsl_path` directly into PyTorch `nn.Parameter` tensors:
- `atom_val` ($\theta_u$): Continuous thresholds (`hcp >= 11.5`, `losing_trick_count <= 6.5`, etc.)
- `rule_atom_logits` ($W_{\text{rule}}$): Conjunction incidence matrix
- `rule_prio_param` & `rule_call_logits`: Differentiable rule priority and call logits
- Auxiliary `partner_belief_net` (5-dim Partner HCP + 4-suit lengths), `value_head` $V(s)$, and 10-strain `dd_pred_head`.

In **Hard mode** (`soft=False`), it matches symbolic `DecisionNet` bit-for-bit. In **Soft STE mode** (`soft=True`), backpropagation computes exact threshold sensitivities $\left|\frac{\partial (-V + 0.25 H(\pi))}{\partial \theta_u}\right|$ to guide optimization — while `explain_hand(...)` always returns a 100% auditable `BidExplanation`!


In [ ]:
diff_net = DiffDSLNet.from_dsl(stage4_dsl_path, device=DEVICE)
print(f"Compiled DiffDSLNet: {diff_net.num_rules} rules, {diff_net.num_atoms} unique condition atoms on [{DEVICE}]")

# Train auxiliary Partner-Belief, State-Value, and 10-Strain DD Heads online
train_sub = jps.slice_bundle(full_bundle, torch.arange(min(512, full_bundle.num_boards), device=DEVICE))
head_stats = jps.train_belief_and_value_heads(
    train_sub,
    diff_net,
    steps=(40 if DEVICE.type == "cuda" else 15),
    batch_boards=min(512, train_sub.num_boards),
)
print(f"Trained Auxiliary Heads on {head_stats['n_decisions']} states: "
      f"belief_mse={head_stats['belief_mse']:.4f}, val_mse={head_stats['val_mse']:.4f}, dd_mse={head_stats['dd_mse']:.4f}")

# One backward pass in soft=True mode reveals top gradient-sensitive rule thresholds
atom_grad, rule_grad = jps.compute_ste_gradient_importance(train_sub, diff_net)
tunable_indices = torch.where(diff_net.atom_tunable_mask)[0].tolist()
tunable_indices.sort(key=lambda u: float(atom_grad[u].item()), reverse=True)
print("\nTop-5 Gradient-Sensitive Rule Thresholds (|∂L / ∂atom_val|):")
for u in tunable_indices[:5]:
    f_key = ga.FEATURE_KEYS[int(diff_net.atom_feat[u].item())]
    val = float(diff_net.atom_val[u].item())
    print(f"  atom[{u:4d}] {f_key:<24s} thresh={val:5.1f} | |grad|={float(atom_grad[u].item()):.4f}")

# Inspect 100% Explainable Bid Output on Board 0
d0 = fresh_deals[0]
exp0 = diff_net.explain_hand(d0.hands[d0.dealer], [], d0.dealer, d0.dealer, d0.vuln)
print(f"\nSample BidExplanation ({d0.dealer.name} hand: {d0.hands[d0.dealer].hcp} HCP):")
print(" ", exp0.to_cot_str())


## Stage 6 — Multi-Generation GPU Self-Improvement Loop (`JPS` + `alterReach` + League Pool)

Now we run the **3-Stage Joint Policy Search (`JPS`) Self-Improvement Loop** ([`research/jps_search.py`](research/jps_search.py)):
1. **Stage 1 (Depth-2 Coupled JPS + `alterReach`)**: Jointly optimizes `(Opener r1 -> c1', Responder r2 -> c2')` pairs in packed mega-batches, discovering partner downstream rules that only become reachable after changing `r1`, ranked by **Empirical Bayes Shrinkage** $\frac{N}{N + \lambda}\overline{\Delta\text{IMP}}$.
2. **Stage 2 (STE Gradient Threshold Shifts)**: Perturbs the top STE-gradient-ranked continuous thresholds $\theta_u \pm 1, \pm 2$ in `DiffDSLNet.atom_val`.
3. **Stage 3 (Partner-Belief, `SuitC` & Fit Splits)**: Clones and splits high-traffic rules on `combined_hcp_min`, `combined_fit_len_est`, `opp_suit_stoppers`, `controls`, and `first_round_controls` / `second_round_controls`.
4. **Multi-Opponent League Validation Gate**: Every accepted edit must improve **both** Train and out-of-sample Validation against **all** opponents in the League Pool, then the updated checkpoint is appended to the League for Generation `g + 1`!

*(On CUDA Colab GPU, this cell runs full `gpu_scale=True` Depth-2 Coupled Search + `5,000` Train / `2,500` Val boards across multiple generations. On CPU/local smoke tests, it automatically scales down so it finishes quickly.)*


In [ ]:
is_cuda = (DEVICE.type == "cuda")
gpu_scale = is_cuda
N = full_bundle.num_boards

if is_cuda and N >= 10000:
    n_tr, n_val, n_te = 5000, 2500, 2500
    num_gens = 3
    run_coupled = True
    min_gain, min_t = 12.0, 1.4
else:
    n_tr = min(200, N // 2)
    n_val = min(100, (N - n_tr) // 2)
    n_te = min(200, N - n_tr - n_val)
    num_gens = 1
    run_coupled = is_cuda
    min_gain, min_t = 3.0, 1.2

tr_b = jps.slice_bundle(full_bundle, torch.arange(0, n_tr, device=DEVICE))
val_b = jps.slice_bundle(full_bundle, torch.arange(n_tr, n_tr + n_val, device=DEVICE))
te_b = jps.slice_bundle(full_bundle, torch.arange(n_tr + n_val, n_tr + n_val + n_te, device=DEVICE))

# Build League Opponent Pool
opp_nets = [ga.TensorDecisionNet.from_dsl(stage4_dsl_path, device=DEVICE)]
for opp_p in ("system/brill_distilled.dsl", "system/brill_distilled_shipdd.dsl"):
    if os.path.exists(opp_p):
        opp_nets.append(ga.TensorDecisionNet.from_dsl(opp_p, device=DEVICE))

print(f"JPS Splits: Train={n_tr}, Val={n_val}, Test={n_te} | Generations={num_gens} | gpu_scale={gpu_scale} | League={len(opp_nets)}")

for gen in range(1, num_gens + 1):
    gt0 = time.perf_counter()
    print(f"\n--- Generation {gen}/{num_gens} (Rules={diff_net.num_rules}) ---")
    hm = jps.train_belief_and_value_heads(tr_b, diff_net, steps=(40 if gpu_scale else 12), batch_boards=min(1024, n_tr))
    print(f"  [Stage 0] belief_mse={hm['belief_mse']:.4f}, val_mse={hm['val_mse']:.4f}, dd_mse={hm['dd_mse']:.4f}")

    if run_coupled:
        c_edits = jps.run_jps_coupled_search_on_diff_net(
            tr_b, val_b, diff_net, opp_nets,
            min_train_gain=min_gain, min_train_t=min_t, lambda_shrink=15.0, gpu_scale=gpu_scale,
        )
    else:
        c_edits = []

    t_edits = jps.search_threshold_shifts_on_diff_net(
        tr_b, val_b, diff_net, opp_nets,
        min_train_gain=min_gain, min_train_t=min_t, lambda_shrink=15.0, gpu_scale=gpu_scale,
    )
    s_edits = jps.search_jps_partner_belief_splits_on_diff_net(
        tr_b, val_b, diff_net, opp_nets,
        min_train_gain=min_gain, min_train_t=min_t, lambda_shrink=15.0, gpu_scale=gpu_scale, gen_idx=gen,
    )
    gen_t = diff_net.to_tensor_decision_net()
    res_gen = ga.gpu_team_match(te_b, gen_t, opp_nets[-1])
    print(f"  [Gen {gen} Done in {time.perf_counter() - gt0:.1f}s] edits={len(c_edits)+len(t_edits)+len(s_edits)} "
          f"(coupled={len(c_edits)}, thresh={len(t_edits)}, splits={len(s_edits)}) | "
          f"Test vs {opp_nets[-1].name}: {res_gen['mean']:+.4f} IMP/bd (t={res_gen['t']:+.2f})")
    opp_nets.insert(0, gen_t)

evolved_dsl_path = os.path.join(WORK_DIR, "scratch_superhuman_jps.dsl")
evolved_pt_path = os.path.join(WORK_DIR, "scratch_superhuman_jps.pt")
n_exp = diff_net.export_dsl(evolved_dsl_path)
torch.save(diff_net.state_dict(), evolved_pt_path)
print(f"\nSaved final DiffDSLNet ({n_exp} rules) -> {evolved_dsl_path} and weights -> {evolved_pt_path}")


## Stage 7 — Test-Time Bayesian Single-Dummy Rollout Search (`BayesianSingleDummyBidder`) & Board Walkthrough

On high-leverage competitive, game, or slam decisions at the table, [`BayesianSingleDummyBidder`](src/bid/bayesian_search.py) upgrades `DiffDSLNet` with **Single-Dummy lookahead**:
1. Infers exact auction HCP & suit-length bounds (`_infer_seat_constraints`) and ranks sampled 39-card opponent/partner completions by vectorized `TensorDecisionNet` past-turn consistency.
2. Rolls out candidate calls via `BatchedAuctionEnv` and evaluates exact `DDSolver` terminal IMPs across the $K$ consistent worlds (`E[ΔIMP]`, `t-stat`).
3. Emits a 100% explainable `BidExplanation` preserving the symbolic rule trace plus the quantitative Single-Dummy EV!


In [ ]:
from bid.bayesian_search import BayesianSingleDummyBidder
from bid.sampling import Deal, PartialState
from bid.models import Seat, Strain
from bid.dds import DDSolver
from bid.brill.convert import hand_pbn

# Wrap the evolved DSL in Program 4 BayesianSingleDummyBidder (or load production system/brill_jps_diff.dsl)
final_dsl_to_play = "system/brill_jps_diff.dsl" if os.path.exists("system/brill_jps_diff.dsl") else evolved_dsl_path
bayes_bidder = BayesianSingleDummyBidder(final_dsl_to_play, num_worlds=8, pool_size=80, device=DEVICE)

# Play one deal showing all 4 hands, 20-entry DD table, and bid-by-bid rule explanations
sample_deals = build_deals(10, seed=7, include_stratified=False)
demo_deal = sample_deals[9]  # Classic slam decision board (Board 10, seed=7)
dd_tbl = DDSolver.solve_dd_table(demo_deal)

print(f"Demo Deal (Dealer={demo_deal.dealer.name}, Vul={demo_deal.vuln}):")
for s in Seat:
    print(f"  {s.name:<5s} ({demo_deal.hands[s].hcp:2d} HCP): {hand_pbn(demo_deal.hands[s])}")
print("  DD Tricks (N/S): " + " | ".join(
    f"{st.name}: N={dd_tbl[(st, Seat.NORTH)]}/S={dd_tbl[(st, Seat.SOUTH)]}"
    for st in (Strain.CLUBS, Strain.DIAMONDS, Strain.HEARTS, Strain.SPADES, Strain.NT)
))

hist = []
curr = demo_deal.dealer
print("\nBid-by-Bid Explainable Auction Trace:")
while len(hist) < 20:
    ps = PartialState(curr, demo_deal.hands[curr], hist, demo_deal.dealer, demo_deal.vuln)
    if ps.is_auction_over():
        break
    call, exp = bayes_bidder.decide_with_explanation(
        demo_deal.hands[curr], hist, my_seat=curr, dealer=demo_deal.dealer, vuln=demo_deal.vuln
    )
    print(f"  Turn {len(hist):2d} ({curr.name[0]}): {str(call):<4s} | {exp.rule_id:<36s} (prio={exp.priority:.0f})")
    hist.append(call)
    curr = Seat((curr.value + 1) % 4)
print("Final Contract:", PartialState(Seat.SOUTH, demo_deal.hands[Seat.SOUTH], hist, demo_deal.dealer, demo_deal.vuln).get_contract())


## Stage 8 — Full Progression Leaderboard & Next Steps

Below we benchmark every milestone side-by-side on the Held-Out `te_b` split using GPU `gpu_team_match`!

### Want to Push Even Further on Colab?
- **From CLI on Colab `L4` / `A100`**:
  ```bash
  PYTHONPATH=src:research python3 research/jps_search.py \
    --bundle data/colab_bundle_10k.pt \
    --dsl system/champion_system.dsl \
    --out-dsl /tmp/colab_bid_scratch/from_scratch_6gen.dsl \
    --generations 6 --gpu-scale --device cuda
  ```
- **Neural Chain-of-Thought (`COTModel`) + `REINFORCE` Self-Play** (Method 10):
  ```bash
  PYTHONPATH=src python3 -m bid.trace_factory --boards 400 --seed 42 --out /tmp/colab_traces.jsonl
  PYTHONPATH=src python3 -m bid.build_cot_dataset /tmp/colab_traces.jsonl --out-dir /tmp/colab_cot_ds
  PYTHONPATH=src python3 -m bid.cot_model train --dataset /tmp/colab_cot_ds/dataset.json --epochs 15 --batch 64 --out /tmp/cot_ckpt.pt
  PYTHONPATH=src python3 -m bid.rl_finetune --boards 32 --epochs 3 --ckpt /tmp/cot_ckpt.pt --out /tmp/cot_rl.pt
  ```
- **Verify Against Live Remote Brill**:
  ```bash
  PYTHONPATH=src:research python3 research/team_match.py \
    --remote-a --b system/brill_jps_diff.dsl --bayes-b --boards 16 --seed 7
  ```


In [ ]:
baseline_ref = ga.TensorDecisionNet.from_dsl(seed_dsl_path, device=DEVICE)
candidates = [
    ("Stage 2: Seed from Scratch (24 rules)", seed_dsl_path),
    ("Stage 3: Seed + GPU Q-Table ID3", stage3_dsl_path),
    ("Stage 4: + P1/P2 Conventions & DAgger", stage4_dsl_path),
    ("Stage 6: + DiffDSLNet GPU JPS", evolved_dsl_path),
]
if os.path.exists("system/champion_system.dsl"):
    candidates.append(("Repo Champion (CPU Flywheel)", "system/champion_system.dsl"))
if os.path.exists("system/brill_distilled.dsl"):
    candidates.append(("Repo Brill Distilled (330k Traces)", "system/brill_distilled.dsl"))
if os.path.exists("system/brill_jps_diff.dsl"):
    candidates.append(("Repo Production (Gen-7 JPS + P1..4)", "system/brill_jps_diff.dsl"))

print(f"GPU Tournament Leaderboard on {te_b.num_boards} Held-Out Boards (vs Scratch Seed [{DEVICE}]):")
print(f"{'System':<42s} | {'Rules':>5s} | {'IMP/bd':>9s} | {'t-stat':>7s} | {'W / L / T':>13s} | {'Time':>7s}")
print("-" * 98)
for label, dsl_p in candidates:
    t_cand = ga.TensorDecisionNet.from_dsl(dsl_p, device=DEVICE)
    r = ga.gpu_team_match(te_b, t_cand, baseline_ref)
    wlt = f"{r['wins']}/{r['losses']}/{r['ties']}"
    print(f"{label:<42s} | {t_cand.num_rules:5d} | {r['mean']:+9.4f} | {r['t']:+7.2f} | {wlt:>13s} | {r['elapsed_ms']:5.1f}ms")
